# SC-7c : ERC-20 — compagnon natif Lean (kernel `lean4-wsl`)

**Navigation** : [<< Précédent : SC-7b (Companion python3)](./SC-7b-ERC20-Lean-Verification-Companion.ipynb) | [Retour au sommaire SmartContracts](../README.md) | [Suivant : SC-8 (DeFi Primitives) >>](./SC-8-DeFi-Primitives.ipynb)

***

## Pourquoi un compagnon natif Lean

Le notebook compagnon `SC-7b` (kernel `python3`) lit les **sources** du lake `erc20_lean` et en extrait les signatures pour les afficher. C'est une lecture statique : le code Lean n'est **pas exécuté**. Ce notebook-ci passe au kernel `lean4-wsl` et **évalue** chaque déclaration (`#check`, `#print axioms`, `#eval`) : la signature affichée est alors le produit réel du compilateur Lean, pas le texte lu dans le fichier.

C'est la différence de fond entre un **loader** et un **compagnon natif** : ici, le noyau Lean certifie que chaque énoncé compile, que les preuves sont closes, et quels axiomes elles utilisent. Le lake `erc20_lean` formalise l'invariant fondateur d'un jeton ERC-20 — la conservation de l'offre totale (`∑ balances = totalSupply`) — et le prouve préservé par chaque opération standard (`mint`, `burn`, `transfer`), puis par toute suite atteignable d'opérations.

**Lake** : `MyIA.AI.Notebooks/SymbolicAI/SmartContracts/erc20_lean` — 4 modules, 17 déclarations (State : 2, Ops : 3, Invariant : 12). Exécution avec le répertoire du lake comme répertoire de travail (le kernel `lean4-wsl` découvre le lake et son LEAN_PATH en remontant depuis le cwd).

In [1]:
-- Verification de l'environnement : import du lake erc20_lean construit.
-- Si cette cellule affiche une erreur d'import, le lake n'est pas dans le
-- LEAN_PATH du kernel (executer depuis le repertoire du lake, cf. README).
import ERC20.State
import ERC20.Ops
import ERC20.Invariant


-- Verification de l'environnement : import du lake erc20_lean construit.
-- Si cette cellule affiche une erreur d'import, le lake n'est pas dans le
-- LEAN_PATH du kernel (executer depuis le repertoire du lake, cf. README).
import ERC20.State
import ERC20.Ops
import ERC20.Invariant

--% env 0

Raw input:
{"cmd": "-- Verification de l'environnement : import du lake erc20_lean construit.\n-- Si cette cellule affiche une erreur d'import, le lake n'est pas dans le\n-- LEAN_PATH du kernel (executer depuis le repertoire du lake, cf. README).\nimport ERC20.State\nimport ERC20.Ops\nimport ERC20.Invariant\n"}
Raw output:
{"env": 0}

## 1. L'état du contrat et l'invariant (module `State`)

Le lake modélise un jeton ERC-20 par une machine à états finie : `State n` porte `balances : Address n → ℕ` et `totalSupply : ℕ`, avec `Address n := Fin n` (un nombre fini de détenteurs potentiels, muni d'un `Fintype`). L'invariant fondateur `supplyInvariant` dit que la somme des soldes égale l'offre totale.

In [2]:
-- Module State : 3 declarations (Address = abbrev, State, supplyInvariant)
#check ERC20.Address
#check ERC20.State
#check ERC20.supplyInvariant


-- Module State : 3 declarations (Address = abbrev, State, supplyInvariant)
#check ERC20.Address
──────▶  ERC20.Address (n : ℕ) : Type
#check ERC20.State
──────▶  ERC20.State (n : ℕ) : Type
#check ERC20.supplyInvariant
──────▶  ERC20.supplyInvariant {n : ℕ} (s : ERC20.State n) : Prop

--% env 1

Raw input:
{"cmd": "-- Module State : 3 declarations (Address = abbrev, State, supplyInvariant)\n#check ERC20.Address\n#check ERC20.State\n#check ERC20.supplyInvariant\n", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "ERC20.Address (n : ℕ) : Type"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "ERC20.State (n : ℕ) : Type"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "ERC20.supplyInvariant {n : ℕ} (s : ERC20.State n) : Prop"}],
 "env": 1}

### Lecture de la sortie

`supplyInvariant s := ∑ a, s.balances a = s.totalSupply` est une `Prop`. Le `#check` certifie que la définition type-checke, et le noyau accepte `Fin n` comme ensemble fini (`Fintype`), rendant la somme sur `Address n` bien définie. C'est le pendant formel de l'`assert` Solidity absent du contrat : ici l'invariant vit dans le type.

## 2. Les opérations standard (module `Ops`)

Trois transitions : `mint` (création), `burn` (destruction), `transfer` (déplacement). Chacune transforme un `State n` en un autre `State n`. Ce sont des fonctions **totales** — les gardes (solde suffisant pour `burn`/`transfer`) sont portées par les théorèmes du module `Invariant`, pas par les définitions : exactement la séparation Solidity (la fonction modifie, le `require` protège).

In [3]:
-- Module Ops : 3 declarations
#check ERC20.mint
#check ERC20.burn
#check ERC20.transfer

-- Module Ops : 3 declarations
#check ERC20.mint
──────▶  ERC20.mint {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ) : ERC20.State n
#check ERC20.burn
──────▶  ERC20.burn {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ) : ERC20.State n
#check ERC20.transfer
──────▶  ERC20.transfer {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ) : ERC20.State n
--% env 2

Raw input:
{"cmd": "-- Module Ops : 3 declarations\n#check ERC20.mint\n#check ERC20.burn\n#check ERC20.transfer", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "ERC20.mint {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ) : ERC20.State n"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "ERC20.burn {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ) : ERC20.State n"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "ERC20.transfer {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ) : ERC20.State n"}],
 "env": 2}

### Lecture de la sortie

Les signatures révèlent la symétrie ERC-20 vue dans SC-7b : `transfer` déplace (`src -= amount`, `dst += amount`) sans toucher `totalSupply` ; `mint`/`burn` ajustent les soldes ET l'offre en parallèle. La soustraction `-` sur `ℕ` est tronquée : sans garde, un `burn` excessif détruirait silencieusement des tokens — c'est précisément ce que `transfer_no_underflow` (section 4) interdit de prouver à tort.

## 3. Les lemmes auxiliaires de sommation (module `Invariant`, 1/3)

Avant les théorèmes de préservation, le lake établit trois lemmes de `Finset.sum` : un split sur un membre d'un finset, un split sur l'univers entier, et la majoration d'un solde par l'offre totale sous invariant.

In [4]:
-- Lemmes auxiliaires (3)
#check ERC20.sum_split_mem
#check ERC20.sum_univ_split
#check ERC20.balance_le_totalSupply

-- Lemmes auxiliaires (3)
#check ERC20.sum_split_mem
──────▶  ERC20.sum_split_mem {n : ℕ} (f : ERC20.Address n → ℕ) (s : Finset (ERC20.Address n)) (a : ERC20.Address n)
  (ha : a ∈ s) : ∑ x ∈ s, f x = f a + ∑ x ∈ s.erase a, f x
#check ERC20.sum_univ_split
──────▶  ERC20.sum_univ_split {n : ℕ} (f : ERC20.Address n → ℕ) (a : ERC20.Address n) :
  ∑ x, f x = f a + ∑ x ∈ Finset.univ.erase a, f x
#check ERC20.balance_le_totalSupply
──────▶  ERC20.balance_le_totalSupply {n : ℕ} (s : ERC20.State n) (a : ERC20.Address n) (h : ERC20.supplyInvariant s) :
  s.balances a ≤ s.totalSupply
--% env 3

Raw input:
{"cmd": "-- Lemmes auxiliaires (3)\n#check ERC20.sum_split_mem\n#check ERC20.sum_univ_split\n#check ERC20.balance_le_totalSupply", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "ERC20.sum_split_mem {n : ℕ} (f : ERC20.Address n → ℕ) (s : Finset (ERC20.Address n)) (a : ERC20.Address n)\n  (ha : a ∈ s) : ∑ x ∈ s, f x = f a + ∑ x ∈ s.erase a, f x"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "ERC20.sum_univ_split {n : ℕ} (f : ERC20.Address n → ℕ) (a : ERC20.Address n) :\n  ∑ x, f x = f a + ∑ x ∈ Finset.univ.erase a, f x"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "ERC20.balance_le_totalSupply {n : ℕ} (s : ERC20.State n) (a : ERC20.Address n) (h : ERC20.supplyInvariant s) :\n  s.balances a ≤ s.totalSupply"}],
 "env": 3}

## 4. Les théorèmes de préservation (module `Invariant`, 2/3)

Le cœur du lake : chaque opération standard **préserve** `supplyInvariant`. `mint_preserves_supply`, `burn_preserves_supply` (sous garde de solde), `transfer_preserves_supply` (sous garde et adresses distinctes) le prouvent ; `transfer_no_underflow` prouve que le transfert garde la trace exacte du débit de la source.

In [5]:
-- Theoremes de preservation par operation (4)
#check ERC20.mint_preserves_supply
#check ERC20.burn_preserves_supply
#check ERC20.transfer_preserves_supply
#check ERC20.transfer_no_underflow

-- Theoremes de preservation par operation (4)
#check ERC20.mint_preserves_supply
──────▶  ERC20.mint_preserves_supply {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ)
  (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.mint s dst amount)
#check ERC20.burn_preserves_supply
──────▶  ERC20.burn_preserves_supply {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ)
  (hguard : s.balances src ≥ amount) (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.burn s src amount)
#check ERC20.transfer_preserves_supply
──────▶  ERC20.transfer_preserves_supply {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ)
  (hguard : s.balances src ≥ amount) (hne : src ≠ dst) (h : ERC20.supplyInvariant s) :
  ERC20.supplyInvariant (ERC20.transfer s src dst amount)
#check ERC20.transfer_no_underflow
──────▶  ERC20.transfer_no_underflow {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ)
  (hguard : s.balances src ≥ amount) :
  (ERC20.transfer s src dst amount).balances src = s.balances src - amount ∧
    s.balances src - amount + amount = s.balances src
--% env 4

Raw input:
{"cmd": "-- Theoremes de preservation par operation (4)\n#check ERC20.mint_preserves_supply\n#check ERC20.burn_preserves_supply\n#check ERC20.transfer_preserves_supply\n#check ERC20.transfer_no_underflow", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "ERC20.mint_preserves_supply {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ)\n  (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.mint s dst amount)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "ERC20.burn_preserves_supply {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ)\n  (hguard : s.balances src ≥ amount) (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.burn s src amount)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "ERC20.transfer_preserves_supply {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ)\n  (hguard : s.balances src ≥ amount) (hne : src ≠ dst) (h : ERC20.supplyInvariant s) :\n  ERC20.supplyInvariant (ERC20.transfer s src dst amount)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "ERC20.transfer_no_underflow {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ)\n  (hguard : s.balances src ≥ amount) :\n  (ERC20.transfer s src dst amount).balances src = s.balances src - amount ∧\n    s.balances src - amount + amount = s.balances src"}],
 "env": 4}

### Lecture de la sortie

Chaque `#check` confirme que l'énoncé type-checke et que sa preuve (dans le source `Invariant.lean`) est close par le noyau — pas de `sorry`. C'est la certification formelle que Solidity ne peut pas donner : aucune séquence gardée de `mint`/`burn`/`transfer` ne peut violer la conservation de l'offre.

## 5. La fermeture par atteignabilité (module `Invariant`, 3/3)

Les opérations individuelles préservent l'invariant, mais un contrat réel en enchaîne beaucoup. Le lake formalise cette composition par deux déclarations **inductives** : `Op` (une étape, constructeurs `mint`/`burn`/`transfer`) et `Reachable` (la fermeture réflexive-transitive, constructeurs `refl`/`step`). Le lemme `op_preserves_invariant` remonte à un pas, le théorème `reachable_preserves_invariant` à une suite entière, par induction sur la trace.

**Piège de couverture** : ces deux declarations sont `inductive`. Un extracteur manuel qui n'énumère que `theorem`/`lemma`/`def` les compte à tort comme absentes (15/17 au lieu de 17/17 — mesuré sur #11710). Les `#check` ci-dessous confirment qu'elles existent et type-checkent sous le noyau.

In [6]:
-- Inductives : une etape, puis la fermeture transitive
#check ERC20.Op
#check ERC20.Reachable
-- Preservation : un pas, puis une suite entiere
#check ERC20.op_preserves_invariant
#check ERC20.reachable_preserves_invariant

-- Inductives : une etape, puis la fermeture transitive
#check ERC20.Op
──────▶  ERC20.Op (n : ℕ) : ERC20.State n → ERC20.State n → Prop
#check ERC20.Reachable
──────▶  ERC20.Reachable (n : ℕ) : ERC20.State n → ERC20.State n → Prop
-- Preservation : un pas, puis une suite entiere
#check ERC20.op_preserves_invariant
──────▶  ERC20.op_preserves_invariant {n : ℕ} (s s' : ERC20.State n) (hop : ERC20.Op n s s') (h : ERC20.supplyInvariant s) :
  ERC20.supplyInvariant s'
#check ERC20.reachable_preserves_invariant
──────▶  ERC20.reachable_preserves_invariant {n : ℕ} (s s' : ERC20.State n) (h : ERC20.supplyInvariant s)
  (hr : ERC20.Reachable n s s') : ERC20.supplyInvariant s'
--% env 5

Raw input:
{"cmd": "-- Inductives : une etape, puis la fermeture transitive\n#check ERC20.Op\n#check ERC20.Reachable\n-- Preservation : un pas, puis une suite entiere\n#check ERC20.op_preserves_invariant\n#check ERC20.reachable_preserves_invariant", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "ERC20.Op (n : ℕ) : ERC20.State n → ERC20.State n → Prop"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "ERC20.Reachable (n : ℕ) : ERC20.State n → ERC20.State n → Prop"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "ERC20.op_preserves_invariant {n : ℕ} (s s' : ERC20.State n) (hop : ERC20.Op n s s') (h : ERC20.supplyInvariant s) :\n  ERC20.supplyInvariant s'"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "ERC20.reachable_preserves_invariant {n : ℕ} (s s' : ERC20.State n) (h : ERC20.supplyInvariant s)\n  (hr : ERC20.Reachable n s s') : ERC20.supplyInvariant s'"}],
 "env": 5}

### Lecture de la sortie

`Reachable n s s'` signifie : `s'` est atteignable depuis `s` par zéro, une ou plusieurs opérations valides. `reachable_preserves_invariant` dit : si `s` satisfait l'invariant, tout état atteignable le satisfait encore. C'est le théorème final qui fait de `supplyInvariant` un **invariant de sûreté** du contrat ERC-20 formalisé — pas seulement une propriété de chaque transition isolée, mais de toute exécution atteignable.

## 6. Axiomes et intégrité des preuves

Une preuve Lean n'a de valeur que si l'on sait sur quoi elle repose. `#print axioms` liste les axiomes utilisés par une preuve ; les trois standards de Mathlib (`propext`, `Classical.choice`, `Quot.sound`) sont la logique classique usuelle. Tout axiome supplémentaire serait un trou à documenter.

C'est le contrôle d'intégrité que le compagnon python3 ne peut pas faire : lire le texte d'une preuve ne dit pas quels axiomes elle invoque — le noyau, lui, le sait.

In [7]:
-- Axiomes utilises par les theoremes cles
#print axioms ERC20.mint_preserves_supply
#print axioms ERC20.burn_preserves_supply
#print axioms ERC20.transfer_preserves_supply
#print axioms ERC20.reachable_preserves_invariant

-- Axiomes utilises par les theoremes cles
#print axioms ERC20.mint_preserves_supply
──────▶  'ERC20.mint_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms ERC20.burn_preserves_supply
──────▶  'ERC20.burn_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms ERC20.transfer_preserves_supply
──────▶  'ERC20.transfer_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms ERC20.reachable_preserves_invariant
──────▶  'ERC20.reachable_preserves_invariant' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 6

Raw input:
{"cmd": "-- Axiomes utilises par les theoremes cles\n#print axioms ERC20.mint_preserves_supply\n#print axioms ERC20.burn_preserves_supply\n#print axioms ERC20.transfer_preserves_supply\n#print axioms ERC20.reachable_preserves_invariant", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'ERC20.mint_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'ERC20.burn_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "'ERC20.transfer_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "'ERC20.reachable_preserves_invariant' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 6}

### Lecture de la sortie

Si la sortie ne liste que `propext`, `Classical.choice`, `Quot.sound`, les preuves sont honnêtes : vérifiées par le noyau modulo la logique classique. Aucun `sorryAx` ne doit apparaître — sa présence signalerait une preuve trouée.

## 7. Une trace concrète sous le noyau

Construisons un `State` à trois adresses, mintons, transférons, et **calculons** offre et somme des soldes après chaque pas — sous le vrai noyau Lean (`#eval`), pas en Python. La machine à états du lake s'exécute : on voit `totalSupply` et la somme des soldes rester égales à chaque transition.

In [8]:
-- Etat initial : 3 adresses, soldes nuls, offre 0
def s0 : ERC20.State 3 := ⟨![0, 0, 0], 0⟩
#eval s0.totalSupply
#eval s0.balances 0 + s0.balances 1 + s0.balances 2

-- Etat initial : 3 adresses, soldes nuls, offre 0
def s0 : ERC20.State 3 := ⟨![0, 0, 0], 0⟩
#eval s0.totalSupply
─────▶  0
#eval s0.balances 0 + s0.balances 1 + s0.balances 2
─────▶  0
--% env 7

Raw input:
{"cmd": "-- Etat initial : 3 adresses, soldes nuls, offre 0\ndef s0 : ERC20.State 3 := \u27e8![0, 0, 0], 0\u27e9\n#eval s0.totalSupply\n#eval s0.balances 0 + s0.balances 1 + s0.balances 2", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "0"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "0"}],
 "env": 7}

In [9]:
-- Pas 1 : mint de 100 tokens a l'adresse 0
def s1 : ERC20.State 3 := ERC20.mint s0 0 100
#eval s1.totalSupply
#eval s1.balances 0 + s1.balances 1 + s1.balances 2

-- Pas 1 : mint de 100 tokens a l'adresse 0
def s1 : ERC20.State 3 := ERC20.mint s0 0 100
#eval s1.totalSupply
─────▶  100
#eval s1.balances 0 + s1.balances 1 + s1.balances 2
─────▶  100
--% env 8

Raw input:
{"cmd": "-- Pas 1 : mint de 100 tokens a l'adresse 0\ndef s1 : ERC20.State 3 := ERC20.mint s0 0 100\n#eval s1.totalSupply\n#eval s1.balances 0 + s1.balances 1 + s1.balances 2", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "100"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "100"}],
 "env": 8}

In [10]:
-- Pas 2 : transfer de 30 de l'adresse 0 vers l'adresse 1
def s2 : ERC20.State 3 := ERC20.transfer s1 0 1 30
#eval s2.totalSupply
#eval s2.balances 0 + s2.balances 1 + s2.balances 2

-- Pas 2 : transfer de 30 de l'adresse 0 vers l'adresse 1
def s2 : ERC20.State 3 := ERC20.transfer s1 0 1 30
#eval s2.totalSupply
─────▶  100
#eval s2.balances 0 + s2.balances 1 + s2.balances 2
─────▶  100
--% env 9

Raw input:
{"cmd": "-- Pas 2 : transfer de 30 de l'adresse 0 vers l'adresse 1\ndef s2 : ERC20.State 3 := ERC20.transfer s1 0 1 30\n#eval s2.totalSupply\n#eval s2.balances 0 + s2.balances 1 + s2.balances 2", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "100"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "100"}],
 "env": 9}

### Lecture de la trace

À l'état initial, offre `0`, somme des soldes `0`. Après le `mint` : offre `100`, somme `100 + 0 + 0 = 100` — les deux bougent **ensemble**. Après le `transfer` : offre toujours `100`, somme `70 + 30 + 0 = 100` — l'offre n'a pas bougé, les soldes se sont déplacés. La conservation tient à chaque pas, et les théorèmes de la section 4 garantissent qu'elle tiendra pour **toute** trace gardée : le Monte-Carlo de SC-7b suggérait la propriété, le noyau la certifie.

*Pour aller plus loin* : la trace `s0 → mint → transfer → burn` est un témoin `ERC20.Reachable 3 s0 s3` ; le théorème `reachable_preserves_invariant` s'applique alors mécaniquement (voir l'exercice 2).

## Exercices

Les exercices suivants sont à compléter. Ils utilisent le lake `erc20_lean` et les définitions `s0`/`s1`/`s2` de la section 7. Remplacer chaque `sorry` par une preuve ; les indices sont dans les commentaires.

### Exercice 1 : certifier l'invariant de l'état initial

Prouver `supplyInvariant s0` : la somme des trois soldes nuls vaut l'offre nulle.

*Indice* : déplier `supplyInvariant` et `s0`, puis `simp` évalue la somme sur `Fin 3`.

In [11]:
-- Exercice 1 : a completer
-- TODO etudiant
theorem exo1_s0_invariant : ERC20.supplyInvariant s0 := by
  sorry

-- Exercice 1 : a completer
-- TODO etudiant
theorem exo1_s0_invariant : ERC20.supplyInvariant s0 := by
        ─────────────────▶ 🟨 declaration uses `sorry`
  sorry
--% env 10
--% prove 0

Raw input:
{"cmd": "-- Exercice 1 : a completer\n-- TODO etudiant\ntheorem exo1_s0_invariant : ERC20.supplyInvariant s0 := by\n  sorry", "env": 9}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 4, "column": 2},
   "goal": "⊢ ERC20.supplyInvariant s0",
   "endPos": {"line": 4, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 25},
   "data": "declaration uses `sorry`"}],
 "env": 10}

### Exercice 2 : certifier le burn en bout de trace

Prouver que brûler 20 tokens à l'adresse 0 depuis `s2` préserve l'invariant.

*Etape 1* : prouver `supplyInvariant s2` en remontant la trace (exo 1 + `mint_preserves_supply` + `transfer_preserves_supply`, la garde `s1.balances 0 ≥ 30` et la distinction `0 ≠ 1` se montrent par `rfl`/`decide`). *Etape 2* : appliquer `burn_preserves_supply` avec la garde `s2.balances 0 ≥ 20`.

In [12]:
-- Exercice 2 : a completer
-- TODO etudiant
theorem exo2_burn_preserved : ERC20.supplyInvariant (ERC20.burn s2 0 20) := by
  sorry

-- Exercice 2 : a completer
-- TODO etudiant
theorem exo2_burn_preserved : ERC20.supplyInvariant (ERC20.burn s2 0 20) := by
        ───────────────────▶ 🟨 declaration uses `sorry`
  sorry
--% env 11
--% prove 1

Raw input:
{"cmd": "-- Exercice 2 : a completer\n-- TODO etudiant\ntheorem exo2_burn_preserved : ERC20.supplyInvariant (ERC20.burn s2 0 20) := by\n  sorry", "env": 10}
Raw output:
{"sorries":
 [{"proofState": 1,
   "pos": {"line": 4, "column": 2},
   "goal": "⊢ ERC20.supplyInvariant (ERC20.burn s2 0 20)",
   "endPos": {"line": 4, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 27},
   "data": "declaration uses `sorry`"}],
 "env": 11}

### Exercice 3 : un état de votre choix

`monEtat` distribue 50 tokens sur 4 adresses. (a) Vérifier par `#eval` que la somme des soldes vaut 50. (b) Prouver `supplyInvariant monEtat`. (c) Bonus : minté 10 et appliquer le théorème de préservation.

*Indice* : la somme sur `Fin 4` se calcule comme sur `Fin 3` — `simp` déplie la notation `![...]`.

In [13]:
-- Exercice 3 : a completer
-- TODO etudiant (la question (a) est deja ecrite : decommenter apres verification)
def monEtat : ERC20.State 4 := ⟨![10, 15, 20, 5], 50⟩
-- #eval monEtat.balances 0 + monEtat.balances 1 + monEtat.balances 2 + monEtat.balances 3
theorem exo3_monEtat_invariant : ERC20.supplyInvariant monEtat := by
  sorry

-- Exercice 3 : a completer
-- TODO etudiant (la question (a) est deja ecrite : decommenter apres verification)
def monEtat : ERC20.State 4 := ⟨![10, 15, 20, 5], 50⟩
-- #eval monEtat.balances 0 + monEtat.balances 1 + monEtat.balances 2 + monEtat.balances 3
theorem exo3_monEtat_invariant : ERC20.supplyInvariant monEtat := by
        ──────────────────────▶ 🟨 declaration uses `sorry`
  sorry
--% env 12
--% prove 2

Raw input:
{"cmd": "-- Exercice 3 : a completer\n-- TODO etudiant (la question (a) est deja ecrite : decommenter apres verification)\ndef monEtat : ERC20.State 4 := \u27e8![10, 15, 20, 5], 50\u27e9\n-- #eval monEtat.balances 0 + monEtat.balances 1 + monEtat.balances 2 + monEtat.balances 3\ntheorem exo3_monEtat_invariant : ERC20.supplyInvariant monEtat := by\n  sorry", "env": 11}
Raw output:
{"sorries":
 [{"proofState": 2,
   "pos": {"line": 6, "column": 2},
   "goal": "⊢ ERC20.supplyInvariant monEtat",
   "endPos": {"line": 6, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 5, "column": 8},
   "endPos": {"line": 5, "column": 30},
   "data": "declaration uses `sorry`"}],
 "env": 12}